# CodeVerse student fine-tune — Gemma 4 12B on AMD Instinct (ROCm)

Teacher → student distillation. The Fireworks teacher (`glm-5p2`) generated a
validated SFT set from CodeVerse's real prompts (`codeverse_theme_sft.jsonl`).
Here we LoRA fine-tune **Gemma 4 12B** — the product spec's intended model — on an
**AMD Instinct MI300X** with ROCm, **push the adapter to the Hugging Face Hub so
it is permanent**, then serve it as an OpenAI-compatible endpoint the CodeVerse
app plugs into with one config switch.

**Run order:** upload this notebook + `codeverse_theme_sft.jsonl`, then Run All.
Paste a Hugging Face **write** token when the login cell prompts. Training is ~1–1.5h.

Serving uses a small custom shim, not vLLM — vLLM does not support Gemma 4's
architecture (`Gemma4UnifiedForConditionalGeneration`).

In [ ]:
# 1) GPU sanity check — ROCm exposes the Instinct GPU through torch.cuda
import torch
print('torch', torch.__version__)
print('gpu available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0))
print('vram GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

In [ ]:
# 2) Fine-tune deps (torch already in the ROCm image — do NOT reinstall it).
#    transformers>=4.57 is required for the Gemma 4 tokenizer format.
%pip install -q "transformers>=4.57" "peft>=0.14" "trl>=0.15" "datasets>=3.0" accelerate huggingface_hub

In [ ]:
# 3) Load + inspect the distillation dataset
import json
from pathlib import Path

DATA = Path('codeverse_theme_sft.jsonl')
rows = [json.loads(line) for line in DATA.read_text(encoding='utf-8').splitlines() if line.strip()]
print('samples:', len(rows))
print('by task:', {t: sum(1 for r in rows if r['task'] == t) for t in {r['task'] for r in rows}})

# hold out 24 samples for the post-train eval
HOLDOUT = 24
eval_rows, train_rows = rows[:HOLDOUT], rows[HOLDOUT:]
print('train:', len(train_rows), 'eval:', len(eval_rows))

In [ ]:
# 3b) Hugging Face login — Gemma is a GATED model. Once, accept the license at
#     https://huggingface.co/google/gemma-4-12b-it (free), create a token with
#     WRITE scope at https://huggingface.co/settings/tokens (write is needed to
#     push the adapter in step 6), and paste it when prompted.
from huggingface_hub import login, whoami
login()
HF_USER = whoami()['name']
print('logged in as', HF_USER)

In [ ]:
# 4) Load base model. Gemma 4 12B fits the MI300X (192GB) in bf16.
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

BASE = 'google/gemma-4-12b-it'

tokenizer = AutoTokenizer.from_pretrained(BASE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.bfloat16, device_map='auto')
print('loaded', BASE)

In [ ]:
# 5) LoRA SFT — rank 16, 2 epochs on ~545 training conversations.
def to_text(row):
    return {'text': tokenizer.apply_chat_template(row['messages'], tokenize=False)}

train_ds = Dataset.from_list([to_text(r) for r in train_rows])

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    peft_config=LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, task_type='CAUSAL_LM',
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj']),
    args=SFTConfig(
        output_dir='codeverse-student-lora',
        num_train_epochs=2,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=1e-4,
        lr_scheduler_type='cosine',
        logging_steps=5,
        bf16=True,
        gradient_checkpointing=True,
        max_length=3072,
        dataset_text_field='text',
        save_strategy='epoch',
        report_to='none',
    ),
)
trainer.train()
print('training done')

In [ ]:
# 6) Push the LoRA adapter to the Hub — PERMANENT. This is the key fix vs the
#    previous run: the model no longer dies with the notebook. Reload later with
#    PeftModel.from_pretrained(base, HUB_REPO). The adapter is small (~tens of MB).
HUB_REPO = f'{HF_USER}/codeverse-student-gemma4-12b-lora'
trainer.model.push_to_hub(HUB_REPO)
tokenizer.push_to_hub(HUB_REPO)
print('adapter pushed ->', 'https://huggingface.co/' + HUB_REPO)

In [ ]:
# 7) Merge LoRA into a standalone folder for fast serving
merged = trainer.model.merge_and_unload()
merged.save_pretrained('codeverse-student-merged')
tokenizer.save_pretrained('codeverse-student-merged')
print('merged model saved -> codeverse-student-merged')

del trainer, model, merged
torch.cuda.empty_cache()

In [ ]:
# 8) Serve the student as an OpenAI-compatible API — CUSTOM SHIM (stdlib only).
#    Runs in a background thread on :8001. vLLM cannot serve Gemma 4's architecture,
#    so we use transformers.generate behind a tiny HTTP server.
import threading, json as _json, time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from transformers import AutoModelForCausalLM, AutoTokenizer

SERVE_DIR = 'codeverse-student-merged'
serve_tok = AutoTokenizer.from_pretrained(SERVE_DIR)
serve_model = AutoModelForCausalLM.from_pretrained(SERVE_DIR, torch_dtype=torch.bfloat16, device_map='auto')
serve_model.eval()

def _generate(messages, temperature=0.7, max_tokens=2048):
    prompt = serve_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = serve_tok(prompt, return_tensors='pt').to(serve_model.device)
    with torch.no_grad():
        out = serve_model.generate(
            **inputs, max_new_tokens=max_tokens,
            do_sample=temperature > 0, temperature=max(temperature, 1e-5),
            pad_token_id=serve_tok.eos_token_id)
    return serve_tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

class _Handler(BaseHTTPRequestHandler):
    def _send(self, code, obj):
        body = _json.dumps(obj).encode()
        self.send_response(code)
        self.send_header('Content-Type', 'application/json')
        self.send_header('Content-Length', str(len(body)))
        self.end_headers(); self.wfile.write(body)
    def do_GET(self):
        if self.path.startswith('/v1/models'):
            self._send(200, {'object': 'list', 'data': [{'id': 'codeverse-student', 'object': 'model'}]})
        else:
            self._send(404, {'error': 'not found'})
    def do_POST(self):
        if self.path.startswith('/v1/chat/completions'):
            n = int(self.headers.get('Content-Length', 0))
            req = _json.loads(self.rfile.read(n) or b'{}')
            txt = _generate(req.get('messages', []), req.get('temperature', 0.7), req.get('max_tokens', 2048))
            self._send(200, {'id': 'chatcmpl-cv', 'object': 'chat.completion', 'model': 'codeverse-student',
                             'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': txt}, 'finish_reason': 'stop'}]})
        else:
            self._send(404, {'error': 'not found'})
    def log_message(self, *a):
        pass

threading.Thread(target=lambda: ThreadingHTTPServer(('0.0.0.0', 8001), _Handler).serve_forever(), daemon=True).start()
time.sleep(2)
print('custom OpenAI-compatible server up on :8001 (model: codeverse-student)')

In [ ]:
# 9) Eval — JSON validity + latency of the AMD-hosted student on held-out prompts
import json as _json, time as _time, urllib.request as _url

def extract_json_object(raw):
    text = raw.strip()
    if text.startswith('```'):
        text = text[text.index('\n') + 1:]
        if text.rstrip().endswith('```'):
            text = text.rstrip()[:-3]
    start = text.find('{')
    if start == -1:
        raise ValueError('no JSON object')
    try:
        data, _ = _json.JSONDecoder().raw_decode(text[start:])
    except _json.JSONDecodeError:
        end = text.rfind('}')
        if end <= start:
            raise ValueError('no JSON object')
        data = _json.loads(text[start:end + 1])
    if not isinstance(data, dict):
        raise ValueError('no JSON object')
    return data

def call_student(messages):
    body = _json.dumps({'model': 'codeverse-student', 'messages': messages,
                        'temperature': 0.7, 'max_tokens': 2048}).encode()
    req = _url.Request('http://localhost:8001/v1/chat/completions', data=body,
                       headers={'Content-Type': 'application/json'})
    with _url.urlopen(req, timeout=180) as resp:
        return _json.loads(resp.read())['choices'][0]['message']['content']

ok, total, latencies = 0, 0, []
for row in eval_rows:
    prompt_messages = row['messages'][:-1]  # drop the teacher's answer
    t0 = _time.time()
    try:
        data = extract_json_object(call_student(prompt_messages))
        if row['task'] == 'profile':
            assert isinstance(data.get('motifs'), list) and data['motifs']
        else:
            assert isinstance(data.get('questions'), list) and 5 <= len(data['questions']) <= 8
        ok += 1
    except Exception as exc:
        print(f'  [fail:{row["task"]}] {type(exc).__name__}')
    latencies.append(_time.time() - t0)
    total += 1

print(f'\nSTUDENT on AMD: {ok}/{total} valid, avg latency {sum(latencies)/len(latencies):.1f}s')

In [ ]:
# 10) Expose :8001 to the CodeVerse droplet via a reverse SSH tunnel.
#     Put the droplet's tunnel private key at /tmp/tunnel_key (chmod 600), then run
#     the printed command in a JupyterLab TERMINAL (not as a blocking cell) so it
#     forwards the droplet's :8001 to this notebook. The droplet .env already sets
#     CODEVERSE_AMD_BASE_URL=http://172.18.0.1:8001/v1 and CODEVERSE_AMD_ENABLED=true.
DROPLET = 'tunnel@192.241.149.132'
print('In a JupyterLab terminal:')
print(f'  ssh -i /tmp/tunnel_key -N -R 0.0.0.0:8001:localhost:8001 {DROPLET}')
print('Verify on the droplet:  curl -s localhost:8001/v1/models   # lists codeverse-student')